[1] 데이터 준비

In [31]:
## 모듈 로딩
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model

In [32]:
# csv 파일 로드
#wine = pd.read_csv('https://bit.ly/wine-date')
stress = pd.read_csv('Stress-Lysis.csv')
stress.head()

,Humidity,Temperature,Step count,Stress Level
0,21.33,90.33,123,1
1,21.41,90.41,93,1
2,27.12,96.12,196,2
3,27.64,96.64,177,2
4,10.87,79.87,87,0


In [33]:
stress.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Humidity      2001 non-null   float64
 1   Temperature   2001 non-null   float64
 2   Step count    2001 non-null   int64  
 3   Stress Level  2001 non-null   int64  
dtypes: float64(2), int64(2)
memory usage: 62.7 KB


In [34]:
stress.describe()

,Humidity,Temperature,Step count,Stress Level
count,2001.000000,2001.000000,2001.000000,2001.000000
mean,20.000000,89.000000,100.141429,1.104448
std,5.777833,5.777833,58.182948,0.771094
min,10.000000,79.000000,0.000000,0.000000
25%,15.000000,84.000000,50.000000,0.000000
50%,20.000000,89.000000,101.000000,1.000000
75%,25.000000,94.000000,150.000000,2.000000
max,30.000000,99.000000,200.000000,2.000000


In [35]:
stress['Stress Level'].min() #0
stress['Stress Level'].max() #2

2

[2] 데이터 전처리

In [36]:
X = stress.drop(['Stress Level'], axis=1)
y = stress['Stress Level']

In [37]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(train_input.shape, test_input.shape)
print(train_target.shape, test_target.shape)

(1600, 3) (401, 3)
(1600,) (401,)


In [38]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_input = scaler.fit_transform(train_input)
test_input = scaler.transform(test_input)

[3]모델 설계

In [39]:
model = Sequential()

model.add(Dense(128, input_dim=train_input.shape[1]))
model.add(LeakyReLU(alpha=0.1))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(32, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(16, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(len(np.unique(train_target)), activation='softmax'))


In [40]:
## 모델 생성과 관련 필수 요소 [다중 분류]
OPTIM_ = Adam(learning_rate=0.001)
LOSS_ = 'sparse_categorical_crossentropy'
METRICS_ = ['accuracy']

In [41]:
EPOCH = 50
BATCH_SIZE = 32

In [42]:
# 모델 컴파일 (분류 문제이므로 categorical_crossentropy 사용)
model.compile(optimizer=OPTIM_, loss=LOSS_, metrics=METRICS_)

In [43]:
model.fit(train_input, train_target, epochs=EPOCH, batch_size=BATCH_SIZE, validation_data=(test_input, test_target))


Epoch 1/50
50/50 [==============================] - 2s 10ms/step - loss: 0.9540 - accuracy: 0.6125 - val_loss: 0.8878 - val_accuracy: 0.5686
Epoch 2/50
50/50 [==============================] - 0s 5ms/step - loss: 0.4397 - accuracy: 0.8375 - val_loss: 0.5642 - val_accuracy: 0.9426
Epoch 3/50
50/50 [==============================] - 0s 4ms/step - loss: 0.3794 - accuracy: 0.8600 - val_loss: 0.3584 - val_accuracy: 0.9626
Epoch 4/50
50/50 [==============================] - 0s 5ms/step - loss: 0.2758 - accuracy: 0.9038 - val_loss: 0.2359 - val_accuracy: 0.9751
Epoch 5/50
50/50 [==============================] - 0s 4ms/step - loss: 0.2345 - accuracy: 0.9250 - val_loss: 0.1511 - val_accuracy: 0.9875
Epoch 6/50
50/50 [==============================] - 0s 4ms/step - loss: 0.2055 - accuracy: 0.9375 - val_loss: 0.1393 - val_accuracy: 0.9701
Epoch 7/50
50/50 [==============================] - 0s 4ms/step - loss: 0.1999 - accuracy: 0.9344 - val_loss: 0.0860 - val_accuracy: 0.9726
Epoch 8/50
50/50 [=

In [44]:
model.save('stress_model_NoBat.h5')

<hr>

[4] 테스트
- Humidity
- Temperature
- Step count
- Stress Level 

In [45]:
from sklearn.preprocessing import StandardScaler
# 저장된 모델 불러오기
stress_model = load_model('stress_model.h5')

def get_user_input():
    humidity = 15 #float(input("Enter your breathing rate: "))
    temperature = 25 #float(input("Enter your body temperature: "))
    step_count = 120 #float(input("Enter your step count: "))
    
    converted_temperature = (temperature * 9/5) + 32 # 썹씨 -> 화씨

    user_data = np.array([[humidity, converted_temperature, step_count]])
    return user_data

def return_res(input):
    scaler = StandardScaler()
    user_input_scaled = scaler.fit_transform(input)

    y_pred = stress_model.predict(user_input_scaled)
    predicted_class = np.argmax(y_pred, axis=1)[0] 
    stress_levels = ["Low", "Medium", "High"] 

    res = ""
    if predicted_class==0:
        res = stress_levels[0]
    elif predicted_class==1:
        res = stress_levels[1]
    elif predicted_class==2:
        res = stress_levels[2]

    return res

user_input = get_user_input()
res = return_res(user_input)

print(res)


OSError: No file or directory found at stress_model.h5

<hr>
[1] 모델 설계

In [24]:
# ## 모듈 로딩
# from tensorflow.keras import Sequential


In [25]:
# model = Sequential()

[2] 모델 생성

In [26]:
# ## 모델 생성과 관련 필수 요소 [회귀]
# OPTIM_ = 'adam'
# LOSS_ = 'mean_squared_erro'
# METRICS_ = ['r2_score']

In [27]:
# ## 모델 생성과 관련 필수 요소 [이진 분류]
# OPTIM_ = 'adam'
# LOSS_ = 'binary_crossentropy'
# METRICS_ = ['binary_accuracy']

In [28]:
# ## 모델 생성과 관련 필수 요소 [다중 분류]
# OPTIM_ = 'adam'
# LOSS_ = 'sparse_categorical_crossentropy'
# METRICS_ = ['sparse_categorical_accuracy']

In [29]:
# model.compile( optimizer = OPTIM_,
#                 loss = LOSS_,
#                 matrics = METRICS_
# )